# Import Libraries

In [ ]:
import re
import numpy as np

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

from gensim.models import Word2Vec

# Load Dataset

In [ ]:
data = fetch_20newsgroups(
    subset="all",
    remove=("headers", "footers", "quotes")
)

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 15076
Testing samples: 3770


# Text Cleaning

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text

X_train = [clean_text(doc) for doc in X_train]
X_test  = [clean_text(doc) for doc in X_test]

# Experiment 1: TF-IDF + Logistic Regression

In [ ]:
tfidf = TfidfVectorizer(
    max_features=20000,
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)

y_pred_1 = lr.predict(X_test_tfidf)

print("Experiment 1: TF-IDF + Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_1))
print(classification_report(y_test, y_pred_1))

Experiment 1: TF-IDF + Logistic Regression
Accuracy: 0.7363395225464191
              precision    recall  f1-score   support

           0       0.64      0.55      0.59       160
           1       0.70      0.70      0.70       195
           2       0.70      0.66      0.68       197
           3       0.68      0.67      0.67       196
           4       0.79      0.69      0.74       193
           5       0.82      0.84      0.83       198
           6       0.77      0.74      0.76       195
           7       0.76      0.76      0.76       198
           8       0.49      0.73      0.59       199
           9       0.84      0.82      0.83       199
          10       0.91      0.88      0.89       200
          11       0.89      0.81      0.85       198
          12       0.68      0.74      0.71       197
          13       0.78      0.82      0.80       198
          14       0.80      0.78      0.79       197
          15       0.72      0.86      0.78       199
         

# EXPERIMENT 2 — Word2Vec CBOW + Logistic Regression


Tokenization

In [ ]:
tokenized_train = [doc.split() for doc in X_train]
tokenized_test  = [doc.split() for doc in X_test]

Train Word2Vec

In [ ]:
w2v_cbow = Word2Vec(
    sentences=tokenized_train,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0  # CBOW
)

Document Vector Function

In [ ]:
def document_vector(doc, model):
    vectors = [model.wv[word] for word in doc if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

Train Classifier

In [ ]:
X_train_cbow = np.array([document_vector(doc, w2v_cbow) for doc in tokenized_train])
X_test_cbow  = np.array([document_vector(doc, w2v_cbow) for doc in tokenized_test])

lr_cbow = LogisticRegression(max_iter=1000)
lr_cbow.fit(X_train_cbow, y_train)

y_pred_2 = lr_cbow.predict(X_test_cbow)

print("Experiment 2: Word2Vec CBOW + Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_2))
print(classification_report(y_test, y_pred_2))

Experiment 2: Word2Vec CBOW + Logistic Regression
Accuracy: 0.43952254641909816
              precision    recall  f1-score   support

           0       0.32      0.27      0.29       160
           1       0.38      0.41      0.39       195
           2       0.50      0.39      0.44       197
           3       0.41      0.42      0.42       196
           4       0.43      0.35      0.38       193
           5       0.56      0.60      0.58       198
           6       0.65      0.65      0.65       195
           7       0.30      0.42      0.35       198
           8       0.36      0.38      0.37       199
           9       0.45      0.42      0.43       199
          10       0.54      0.56      0.55       200
          11       0.52      0.58      0.55       198
          12       0.38      0.32      0.35       197
          13       0.44      0.49      0.47       198
          14       0.41      0.48      0.44       197
          15       0.53      0.72      0.61       199
 

# EXPERIMENT 3 — Word2Vec Skip-gram + Logistic Regression

Train Skip-gram

In [ ]:
w2v_skipgram = Word2Vec(
    sentences=tokenized_train,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1  # Skip-gram
)

Train Classifier

In [ ]:
X_train_skip = np.array([document_vector(doc, w2v_skipgram) for doc in tokenized_train])
X_test_skip  = np.array([document_vector(doc, w2v_skipgram) for doc in tokenized_test])

lr_skip = LogisticRegression(max_iter=1000)
lr_skip.fit(X_train_skip, y_train)

y_pred_3 = lr_skip.predict(X_test_skip)

print("Experiment 3: Word2Vec Skip-gram + Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_3))
print(classification_report(y_test, y_pred_3))

Experiment 3: Word2Vec Skip-gram + Logistic Regression
Accuracy: 0.5726790450928382
              precision    recall  f1-score   support

           0       0.34      0.33      0.33       160
           1       0.53      0.52      0.52       195
           2       0.54      0.47      0.50       197
           3       0.55      0.51      0.53       196
           4       0.57      0.48      0.52       193
           5       0.70      0.76      0.73       198
           6       0.74      0.69      0.71       195
           7       0.38      0.59      0.46       198
           8       0.45      0.46      0.46       199
           9       0.68      0.60      0.64       199
          10       0.70      0.73      0.72       200
          11       0.72      0.70      0.71       198
          12       0.59      0.53      0.56       197
          13       0.65      0.71      0.68       198
          14       0.64      0.70      0.67       197
          15       0.56      0.79      0.66       1

# TF-IDF Weighted Word2Vec (Skip-gram)

TF-IDF Vocabulary

In [ ]:
tfidf_weight = TfidfVectorizer(
    max_features=20000,
    stop_words="english"
)

tfidf_weight.fit(X_train)

idf_scores = dict(zip(
    tfidf_weight.get_feature_names_out(),
    tfidf_weight.idf_
))

Weighted Document Vector

In [ ]:
def tfidf_weighted_doc_vector(doc, model, idf_scores):
    vectors = []
    weights = []

    for word in doc:
        if word in model.wv and word in idf_scores:
            vectors.append(model.wv[word])
            weights.append(idf_scores[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.average(vectors, axis=0, weights=weights)

 Train Classifier

In [ ]:
X_train_weighted = np.array([
    tfidf_weighted_doc_vector(doc, w2v_skipgram, idf_scores)
    for doc in tokenized_train
])

X_test_weighted = np.array([
    tfidf_weighted_doc_vector(doc, w2v_skipgram, idf_scores)
    for doc in tokenized_test
])

lr_weighted = LogisticRegression(max_iter=1000)
lr_weighted.fit(X_train_weighted, y_train)

y_pred_4 = lr_weighted.predict(X_test_weighted)

print("Experiment 4: TF-IDF Weighted Word2Vec (Skip-gram)")
print("Accuracy:", accuracy_score(y_test, y_pred_4))
print(classification_report(y_test, y_pred_4))

Experiment 4: TF-IDF Weighted Word2Vec (Skip-gram)
Accuracy: 0.6129973474801061
              precision    recall  f1-score   support

           0       0.39      0.37      0.38       160
           1       0.60      0.56      0.58       195
           2       0.57      0.52      0.54       197
           3       0.58      0.52      0.55       196
           4       0.55      0.52      0.54       193
           5       0.74      0.77      0.75       198
           6       0.71      0.67      0.69       195
           7       0.41      0.68      0.51       198
           8       0.53      0.50      0.52       199
           9       0.71      0.70      0.71       199
          10       0.80      0.78      0.79       200
          11       0.77      0.75      0.76       198
          12       0.62      0.60      0.61       197
          13       0.72      0.73      0.72       198
          14       0.71      0.75      0.73       197
          15       0.61      0.76      0.68       199
 

Summary

In [ ]:
print("FINAL ACCURACY SUMMARY")
print("----------------------")
print("Experiment 1 (TF-IDF):", accuracy_score(y_test, y_pred_1))
print("Experiment 2 (CBOW):", accuracy_score(y_test, y_pred_2))
print("Experiment 3 (Skip-gram):", accuracy_score(y_test, y_pred_3))
print("Experiment 4 (TF-IDF + Skip-gram):", accuracy_score(y_test, y_pred_4))

FINAL ACCURACY SUMMARY
----------------------
Experiment 1 (TF-IDF): 0.7363395225464191
Experiment 2 (CBOW): 0.43952254641909816
Experiment 3 (Skip-gram): 0.5726790450928382
Experiment 4 (TF-IDF + Skip-gram): 0.6129973474801061


## Summary and Conclusion

 In this notebook, a multi-class text classification task was performed using the 20 Newsgroups dataset. Four experiments were designed to compare traditional text representations with word embedding-based approaches under a consistent evaluation framework.

The first experiment used TF-IDF features with Logistic Regression as a strong traditional baseline, achieving the highest accuracy due to its ability to capture word importance and class-specific terms. The second and third experiments applied Word2Vec embeddings using CBOW and Skip-gram architectures, respectively. Document representations were created by averaging word embeddings, which resulted in lower accuracy due to the loss of word frequency and contextual importance. Skip-gram outperformed CBOW, particularly in handling rare and informative words.

The final experiment combined TF-IDF weighting with Word2Vec Skip-gram embeddings to construct weighted document vectors. This approach significantly improved performance compared to simple averaging, demonstrating that preserving word importance alongside semantic representations leads to more discriminative document features.

Overall, the results show that while word embeddings provide meaningful semantic representations, traditional TF-IDF remains a strong baseline for topic-based text classification. Embedding-based methods benefit from appropriate aggregation strategies and task-specific weighting to achieve competitive performance.